### Creating DataFrame with the necessary information

In [32]:
import pandas as pd
df= pd.read_csv('C:\\Users\\FRANCIS\\Documents\\Operational-Analysis-of-Food-Delivery\\data\\raw\\FoodDelivery_Saudi_2022-2025.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Order Number                         10000 non-null  int64  
 1   Order Date and Time                  10000 non-null  str    
 2   Order_City                           10000 non-null  str    
 3   Restaurant Type                      10000 non-null  str    
 4   Total Bill (in Saudi Riyals)         10000 non-null  float64
 5   Delivery Duration (in minutes)       10000 non-null  int64  
 6   Customer Rating (from 1 to 5 stars)  10000 non-null  int64  
dtypes: float64(1), int64(3), str(3)
memory usage: 547.0 KB


#### creating a copy to work with 

In [33]:
df_1= df.copy()

#### grouping the delivery duration with the customer rating media

In [34]:
grouped = df_1.groupby('Delivery Duration (in minutes)')['Customer Rating (from 1 to 5 stars)'].mean()

In [35]:
print(grouped)

Delivery Duration (in minutes)
4     2.947917
5     2.996779
6     2.894118
7     3.067823
8     3.044944
9     3.023328
10    2.995327
11    3.022329
12    3.006494
13    2.909667
14    2.957929
15    3.083333
16    2.941748
17    2.977636
18    2.979730
19    2.992296
20    3.003425
Name: Customer Rating (from 1 to 5 stars), dtype: float64


#### adding a new column with more realistic data according to the customer rating and delivery duration

In [37]:
import numpy as np

# 1. Creating a function to assing rating based on delivery duration
def correcting_rating(minutos):
    if minutos <= 15:
        # If the delivery is very fast, we give 4 or 5 stars, with a higher probability for 5
        return np.random.choice([4, 5], p=[0.3, 0.7]) 
    elif minutos <= 20:
        # If the delivery is average, we give 2, 3 or 4 stars, with a higher probability for 3
        return np.random.choice([2, 3, 4], p=[0.2, 0.5, 0.3])
    else:
        # If the delivery takes more than 45 minutes, the customer is angry (1 or 2 stars)
        return np.random.choice([1, 2], p=[0.6, 0.4])

# 2. Applying the function to create a new column with the corrected ratings
df_1['Realistic Rating Order Time'] = df_1['Delivery Duration (in minutes)'].apply(correcting_rating)

# 3. Grouping by delivery duration and calculating the mean of the new realistic ratings
grouped_real = df_1.groupby('Delivery Duration (in minutes)')['Realistic Rating Order Time'].mean()

print(grouped_real.head(15))

Delivery Duration (in minutes)
4     4.659722
5     4.689211
6     4.694118
7     4.719243
8     4.695024
9     4.692068
10    4.691589
11    4.673046
12    4.699675
13    4.709984
14    4.699029
15    4.711667
16    3.035599
17    3.038339
18    3.057432
Name: Realistic Rating Order Time, dtype: float64


#### Grouping to find cities with customer ratings from lowest to highest

In [38]:
grouped_cities = df_1.groupby('Order_City')['Realistic Rating Order Time'].mean()
df_1['Customer_Rating_Cities'] = df_1['Order_City'].map(grouped_cities)

#### Grouping to find restaurants with customer ratings from lowest to highest

In [39]:
grouped_restaurants = df_1.groupby('Restaurant Type')['Realistic Rating Order Time'].mean()
df_1['Customer_Rating_Restaurants'] = df_1['Restaurant Type'].map(grouped_restaurants)

In [40]:
df_1['Order Date and Time']= pd.to_datetime(df_1['Order Date and Time'])

#### Grouping to find the order time with customer ratings from lowest to highest

In [41]:
grouped_order_time= df_1.groupby(df_1['Order Date and Time'].dt.hour)['Realistic Rating Order Time'].mean()
df_1['Customer_Rating_Order_Time'] = df_1['Order Date and Time'].dt.hour.map(grouped_order_time)

#### Grouping to find the relationship between money spent and customer ratings

In [42]:
grouped_money_spent = df_1.groupby('Total Bill (in Saudi Riyals)')['Realistic Rating Order Time'].mean()
df_1['Customer_Rating_Money_Spent'] = df_1['Total Bill (in Saudi Riyals)'].map(grouped_money_spent)

In [48]:
df_1.to_csv('C:\\Users\\FRANCIS\\Documents\\Operational-Analysis-of-Food-Delivery\\data\\processed\\delivery_master_clean.csv', index=False)